# Reporte de estructura NetCDF

Seleccione el archivo NetCDF en la primera línea de la celda siguiente y ejecútela. La salida resume la estructura, el dominio, la cobertura temporal, los niveles verticales y las variables incluidas.

In [11]:
from datetime import datetime
from pathlib import Path

import numpy as np
from netCDF4 import Dataset, num2date


# Modificar únicamente esta ruta para evaluar otro producto NetCDF.
NETCDF_FILE = Path(
    "/workspace/data/standardized/few_variables/ecmwf-ifs_20260903_00Z.nc"
)


def attribute(source, name, default="-"):
    """Obtiene un atributo NetCDF sin interrumpir el reporte si no existe."""
    return getattr(source, name, default)


def as_array(values):
    """Convierte valores NetCDF, incluso arreglos enmascarados, en un arreglo NumPy."""
    return np.asarray(np.ma.filled(values, np.nan), dtype=float)


def coordinate_resolution(values):
    values = as_array(values)
    if len(values) < 2:
        return None
    return float(np.nanmedian(np.abs(np.diff(values))))


def format_value(value, decimals=2):
    return f"{value:.{decimals}f}"


def format_datetime(value):
    return value.strftime("%Y-%m-%d %H:%M UTC")


def parse_reference_time(value):
    if not isinstance(value, str) or value == "-":
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).replace(tzinfo=None)
    except ValueError:
        return None


def frequency_summary(dates):
    if len(dates) < 2:
        return "No aplica"

    intervals = np.array([
        (dates[index + 1] - dates[index]).total_seconds() / 3600
        for index in range(len(dates) - 1)
    ])
    unique, counts = np.unique(np.round(intervals, 6), return_counts=True)
    if len(unique) == 1:
        return f"cada {unique[0]:g} h"
    return "; ".join(
        f"cada {step:g} h ({count} intervalos)"
        for step, count in zip(unique, counts)
    )


def format_levels(levels, units):
    levels = sorted(as_array(levels), reverse=True)
    shown = levels if len(levels) <= 8 else levels[:6] + [np.nan] + [levels[-1]]
    values = ["..." if np.isnan(value) else f"{value:g}" for value in shown]
    return f"{', '.join(values)} {units}"


def longitude_convention(dataset, longitudes):
    declared = attribute(dataset, "longitude_convention")
    if declared != "-":
        return {"0_360": "0 a 360 grados", "-180_180": "-180 a 180 grados"}.get(declared, declared)
    return "0 a 360 grados" if np.nanmin(longitudes) >= 0 else "-180 a 180 grados"


if not NETCDF_FILE.is_file():
    raise FileNotFoundError(f"No se encontró el archivo NetCDF: {NETCDF_FILE}")

with Dataset(NETCDF_FILE) as dataset:
    print("=" * 60)
    print("REPORTE DE ESTRUCTURA NETCDF")
    print("=" * 60)

    print("Archivo:")
    print(f"  {NETCDF_FILE.name}")

    reference_time = attribute(dataset, "reference_time")
    print("Identificación:")
    print(f"  Modelo: {str(attribute(dataset, 'model')).upper()}")
    print(f"  Centro productor: {str(attribute(dataset, 'center')).upper()}")
    print(f"  Tiempo de referencia: {reference_time}")
    print(f"  Formato: {dataset.data_model}")

    print("Estructura y dominio:")
    print("  Dimensiones:")
    for name, dimension in dataset.dimensions.items():
        print(f"    {name}: {len(dimension)}")

    latitude = dataset.variables.get("latitude")
    longitude = dataset.variables.get("longitude")
    if latitude is not None and longitude is not None:
        latitudes = as_array(latitude[:])
        longitudes = as_array(longitude[:])
        lat_resolution = coordinate_resolution(latitudes)
        lon_resolution = coordinate_resolution(longitudes)

        print("  Malla:")
        print(f"    Tamaño: {len(latitudes)} × {len(longitudes)} puntos")
        print("    Coordenadas: latitude, longitude")
        print("    Dominio:")
        print(f"      Latitud: {format_value(np.nanmin(latitudes))} a {format_value(np.nanmax(latitudes))} grados norte")
        print(f"      Longitud: {format_value(np.nanmin(longitudes))} a {format_value(np.nanmax(longitudes))} grados este")
        print(f"    Convención de longitudes: {longitude_convention(dataset, longitudes)}")
        print("    Resolución aproximada:")
        print(f"      Latitud: {format_value(lat_resolution, 4)} grados" if lat_resolution is not None else "      Latitud: no disponible")
        print(f"      Longitud: {format_value(lon_resolution, 4)} grados" if lon_resolution is not None else "      Longitud: no disponible")

    time = dataset.variables.get("time")
    if time is not None and len(time) > 0:
        dates = np.atleast_1d(num2date(
            time[:],
            units=time.units,
            calendar=attribute(time, "calendar", "standard"),
            only_use_cftime_datetimes=False,
            only_use_python_datetimes=True,
        ))
        initial_reference = parse_reference_time(reference_time)
        start = dates[0]
        end = dates[-1]
        horizon_start = initial_reference if initial_reference is not None else start
        horizon = (end - horizon_start).total_seconds() / 3600

        print("Cobertura temporal:")
        print(f"  Número de tiempos: {len(dates)}")
        print(f"  Inicio: {format_datetime(start)}")
        print(f"  Fin: {format_datetime(end)}")
        print(f"  Horizonte de pronóstico: {horizon:g} horas")
        print(f"  Frecuencia temporal: {frequency_summary(dates)}")

    levels = dataset.variables.get("lev")
    if levels is not None:
        level_units = attribute(levels, "units")
        displayed_units = "hPa" if level_units == "millibars" else level_units
        print("Niveles verticales:")
        print(f"  Número de niveles: {len(levels)}")
        print("  Tipo: presión")
        print(f"  Unidades: {displayed_units}")
        print(f"  Niveles: {format_levels(levels[:], displayed_units)}")

    coordinate_names = {"time", "lev", "latitude", "longitude"}
    variables = [
        (name, variable) for name, variable in dataset.variables.items()
        if name not in coordinate_names
    ]

    print("Variables incluidas:")
    header = (
        f"{'Variable':<14} {'Dimensiones':<39} {'Unidades':<10} "
        f"{'Firma':<10} {'Nombre GRIB2':<16} Descripción"
    )
    print("  " + header)
    for name, variable in variables:
        dimensions = ", ".join(variable.dimensions)
        units = str(attribute(variable, "units"))
        signature = str(attribute(variable, "parameter_signature"))
        grib_name = str(attribute(variable, "wgrib2_short_name"))
        description = str(attribute(variable, "long_name"))
        print(
            f"  {name:<14} {dimensions:<39} {units:<10} "
            f"{signature:<10} {grib_name:<16} {description}"
        )

    print("=" * 120)


REPORTE DE ESTRUCTURA NETCDF
Archivo:
  ecmwf-ifs_20260903_00Z.nc
Identificación:
  Modelo: IFS
  Centro productor: ECMWF
  Tiempo de referencia: 2026-09-03T00:00:00
  Formato: NETCDF4
Estructura y dominio:
  Dimensiones:
    time: 65
    lev: 13
    latitude: 349
    longitude: 481
  Malla:
    Tamaño: 349 × 481 puntos
    Coordenadas: latitude, longitude
    Dominio:
      Latitud: -57.00 a 30.00 grados norte
      Longitud: 230.00 a 350.00 grados este
    Convención de longitudes: 0 a 360 grados
    Resolución aproximada:
      Latitud: 0.2500 grados
      Longitud: 0.2500 grados
Cobertura temporal:
  Número de tiempos: 65
  Inicio: 2026-09-03 00:00 UTC
  Fin: 2026-09-13 00:00 UTC
  Horizonte de pronóstico: 240 horas
  Frecuencia temporal: cada 3 h (48 intervalos); cada 6 h (16 intervalos)
Niveles verticales:
  Número de niveles: 13
  Tipo: presión
  Unidades: hPa
  Niveles: 1000, 925, 850, 700, 600, 500, ..., 50 hPa
Variables incluidas:
  Variable       Dimensiones                 